# TriadLM #4 — align the 50M base (Kaggle free T4)

SFT → constitutional triples + CSFT → DPO → eval all variants + grounded.
Needs: **GPU T4 ON**, **Internet ON**, `triadlm-m1-50m` on the Hub (notebook #3 done).
~1h. Every step writes a checkpoint; resume anywhere by re-running that cell.

In [ ]:
!pip install -q torch tokenizers pyyaml tqdm "pydantic>=2" requests huggingface_hub
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-ONLY — enable GPU!")

In [ ]:
!git clone https://github.com/PhillipMtalika/triadlm.git
%cd triadlm
!pwd && ls
!python -m pytest tests/test_tokenizer.py tests/test_model.py -q 2>&1 | tail -n 1

In [ ]:
# Pull the M1 base + tokenizer from the Hub (no retraining).
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import HfApi
me = HfApi(token=token).whoami(token)["name"]
import shutil, os, glob as _g
dst = snapshot_download(repo_id=f"{me}/triadlm-m1-50m", token=token)
os.makedirs("data/checkpoints/m1base", exist_ok=True)
os.makedirs("data/checkpoints/base_50m/tokenizer", exist_ok=True)
d2 = snapshot_download(repo_id=f"{me}/triadlm-corpus-v2", repo_type="dataset", token=token)
base_ckpt = None
for f in _g.glob(os.path.join(dst, "*.pt")):
    if "final" in f:
        shutil.copy(f, "data/checkpoints/m1base/base.pt")
        base_ckpt = "data/checkpoints/m1base/base.pt"
for f in _g.glob(os.path.join(d2, "tokenizer", "*")):
    shutil.copy(f, "data/checkpoints/base_50m/tokenizer/")
print("base:", base_ckpt)

In [ ]:
# SFT on demonstrations (masked loss: only response tokens count).
!python -m triadlm.finetune_sft --config configs/m3_50m.yaml --checkpoint data/checkpoints/m1base/base.pt --out data/checkpoints/m1align/sft.pt --steps 100
!python -m evals.run_eval --checkpoint data/checkpoints/m1align/sft.pt --variant sft --out experiments/runs/m1-sft.json 2>&1 | grep -E '"(run_id|factuality|instruction_following|over_refusal_rate|chichewa_gap)"'

In [ ]:
# Constitutional triples from the REAL base generator, then CSFT.
import json
from triadlm.generate import load_for_inference
from triadlm.constitutional import load_principles, build_constitutional_dataset, train_constitutional_sft
model, tok = load_for_inference("data/checkpoints/m1align/sft.pt")
principles = load_principles("docs/constitution.md")
prompts = [json.loads(l)["prompt"] for l in open("evals/probes.jsonl") if json.loads(l)["category"] in ("knowledge", "citation", "privacy", "prompt_injection", "arithmetic")][:20]
ds = build_constitutional_dataset(prompts, principles, model, tok, source="synthetic")
open("data/constitutional_m1.jsonl", "w").write("\n".join(json.dumps(r) for r in ds))
print("triples:", len(ds))
train_constitutional_sft("data/checkpoints/m1align/sft.pt", "data/constitutional_m1.jsonl", "data/checkpoints/m1align/constitutional_out")
print("csft done")

In [ ]:
!python -m evals.run_eval --checkpoint data/checkpoints/m1align/constitutional_out/constitutional.pt --variant constitutional --out experiments/runs/m1-constitutional.json 2>&1 | grep -E '"(run_id|factuality|instruction_following|over_refusal_rate|chichewa_gap)"'

In [ ]:
# DPO on preferences, then eval + grounded eval off the DPO model.
import triadlm.preference as P
P.train_dpo("data/checkpoints/m1align/constitutional_out/constitutional.pt", "data/checkpoints/m1align/sft.pt", "data/preferences.jsonl", "data/checkpoints/m1align/dpo_out")
print("dpo done")

In [ ]:
!python -m evals.run_eval --checkpoint data/checkpoints/m1align/dpo_out/dpo.pt --variant dpo --out experiments/runs/m1-dpo.json 2>&1 | grep -E '"(run_id|factuality|instruction_following|over_refusal_rate|chichewa_gap)"'
!python -m evals.run_eval --checkpoint data/checkpoints/m1align/dpo_out/dpo.pt --variant grounded --out experiments/runs/m1-grounded.json 2>&1 | grep -E '"(run_id|factuality|citation_accuracy|over_refusal_rate|refusal_precision|chichewa_gap)"'

In [ ]:
# Publish all four branches into the same model repo.
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)
me = api.whoami(token)["name"]
api.upload_folder(folder_path="data/checkpoints/m1align", repo_id=f"{me}/triadlm-m1-50m", token=token)
print("uploaded align")